In [1]:
!pip -q install prefixspan

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [2]:
import pandas as pd
import json
import numpy as np

FILE_PATH = "/content/activation_state_sequences.csv"

df = pd.read_csv(FILE_PATH)

# CSV-তে sequence string হিসেবে আছে, আবার list বানাচ্ছি
df["State_Sequence"] = df["State_Sequence"].apply(json.loads)

print("Dataset shape:", df.shape)
print("Total sequences:", len(df))
print("Unique prompts:", df["Prompt_ID"].nunique())

print("\nColumns:")
print(df.columns.tolist())

print("\nLabels:")
print(
    df[["Prompt_ID", "Response_Label"]]
    .drop_duplicates()["Response_Label"]
    .value_counts()
    .sort_index()
)

print("\nLayers:")
print(df["Layer_ID"].value_counts().sort_index())

print("\nExample sequence:")
print(df.iloc[0]["State_Sequence"][:20])

Dataset shape: (150, 5)
Total sequences: 150
Unique prompts: 50

Columns:
['Prompt_ID', 'Response_Label', 'Layer_ID', 'State_Sequence', 'Sequence_Length']

Labels:
Response_Label
0    25
1    25
Name: count, dtype: int64

Layers:
Layer_ID
10    50
18    50
26    50
Name: count, dtype: int64

Example sequence:
[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]


In [3]:
from prefixspan import PrefixSpan
import pandas as pd

LAYER = 10

# Separate truthful and hallucinated sequences
truthful_sequences = df[
    (df["Layer_ID"] == LAYER) &
    (df["Response_Label"] == 0)
]["State_Sequence"].tolist()

hallucinated_sequences = df[
    (df["Layer_ID"] == LAYER) &
    (df["Response_Label"] == 1)
]["State_Sequence"].tolist()

print("Layer:", LAYER)
print("Truthful sequences:", len(truthful_sequences))
print("Hallucinated sequences:", len(hallucinated_sequences))

# PrefixSpan
truthful_ps = PrefixSpan(truthful_sequences)
hallucinated_ps = PrefixSpan(hallucinated_sequences)

# Pattern must appear in at least 20% of responses
MIN_SUPPORT = 5   # 20% of 25

truthful_patterns = truthful_ps.frequent(MIN_SUPPORT)
hallucinated_patterns = hallucinated_ps.frequent(MIN_SUPPORT)

# Keep useful short patterns only: length 2–5
truthful_patterns = [
    (support, pattern)
    for support, pattern in truthful_patterns
    if 2 <= len(pattern) <= 5
]

hallucinated_patterns = [
    (support, pattern)
    for support, pattern in hallucinated_patterns
    if 2 <= len(pattern) <= 5
]

# Sort by support
truthful_patterns.sort(reverse=True, key=lambda x: x[0])
hallucinated_patterns.sort(reverse=True, key=lambda x: x[0])

print("\nTruthful patterns found:", len(truthful_patterns))
print("Hallucinated patterns found:", len(hallucinated_patterns))

print("\nTop 10 Truthful patterns:")
for support, pattern in truthful_patterns[:10]:
    print(pattern, "Support =", support)

print("\nTop 10 Hallucinated patterns:")
for support, pattern in hallucinated_patterns[:10]:
    print(pattern, "Support =", support)

Layer: 10
Truthful sequences: 25
Hallucinated sequences: 25

Truthful patterns found: 4
Hallucinated patterns found: 4

Top 10 Truthful patterns:
[3, 3] Support = 25
[3, 3, 3] Support = 25
[3, 3, 3, 3] Support = 25
[3, 3, 3, 3, 3] Support = 25

Top 10 Hallucinated patterns:
[3, 3] Support = 25
[3, 3, 3] Support = 25
[3, 3, 3, 3] Support = 25
[3, 3, 3, 3, 3] Support = 24


In [4]:
from prefixspan import PrefixSpan

MIN_SUPPORT = 5

for layer in [18, 26]:

    truthful_sequences = df[
        (df["Layer_ID"] == layer) &
        (df["Response_Label"] == 0)
    ]["State_Sequence"].tolist()

    hallucinated_sequences = df[
        (df["Layer_ID"] == layer) &
        (df["Response_Label"] == 1)
    ]["State_Sequence"].tolist()

    truthful_ps = PrefixSpan(truthful_sequences)
    hallucinated_ps = PrefixSpan(hallucinated_sequences)

    truthful_patterns = truthful_ps.frequent(MIN_SUPPORT)
    hallucinated_patterns = hallucinated_ps.frequent(MIN_SUPPORT)

    truthful_patterns = [
        (support, pattern)
        for support, pattern in truthful_patterns
        if 2 <= len(pattern) <= 5
    ]

    hallucinated_patterns = [
        (support, pattern)
        for support, pattern in hallucinated_patterns
        if 2 <= len(pattern) <= 5
    ]

    truthful_patterns.sort(
        reverse=True,
        key=lambda x: x[0]
    )

    hallucinated_patterns.sort(
        reverse=True,
        key=lambda x: x[0]
    )

    print("\n======================")
    print("LAYER:", layer)
    print("======================")

    print("Truthful patterns found:",
          len(truthful_patterns))

    print("Hallucinated patterns found:",
          len(hallucinated_patterns))

    print("\nTop 10 Truthful:")
    for support, pattern in truthful_patterns[:10]:
        print(pattern, "Support =", support)

    print("\nTop 10 Hallucinated:")
    for support, pattern in hallucinated_patterns[:10]:
        print(pattern, "Support =", support)


LAYER: 18
Truthful patterns found: 4
Hallucinated patterns found: 4

Top 10 Truthful:
[3, 3] Support = 25
[3, 3, 3] Support = 25
[3, 3, 3, 3] Support = 25
[3, 3, 3, 3, 3] Support = 25

Top 10 Hallucinated:
[3, 3] Support = 25
[3, 3, 3] Support = 25
[3, 3, 3, 3] Support = 25
[3, 3, 3, 3, 3] Support = 24


KeyboardInterrupt: 

In [5]:
from prefixspan import PrefixSpan

LAYER = 26
MIN_SUPPORT = 5
MAX_PATTERN_LENGTH = 5

truthful_sequences = df[
    (df["Layer_ID"] == LAYER) &
    (df["Response_Label"] == 0)
]["State_Sequence"].tolist()

hallucinated_sequences = df[
    (df["Layer_ID"] == LAYER) &
    (df["Response_Label"] == 1)
]["State_Sequence"].tolist()


def mine_safe(sequences):

    ps = PrefixSpan(sequences)

    # IMPORTANT:
    # Limit pattern length DURING mining
    ps.maxlen = MAX_PATTERN_LENGTH

    patterns = ps.frequent(MIN_SUPPORT)

    # We only need length 2–5
    patterns = [
        (support, pattern)
        for support, pattern in patterns
        if 2 <= len(pattern) <= MAX_PATTERN_LENGTH
    ]

    patterns.sort(
        key=lambda x: x[0],
        reverse=True
    )

    return patterns


truthful_26 = mine_safe(truthful_sequences)
hallucinated_26 = mine_safe(hallucinated_sequences)

print("===== LAYER 26 =====")

print("\nTruthful patterns found:", len(truthful_26))
print("Hallucinated patterns found:", len(hallucinated_26))

print("\nTop 10 Truthful:")
for support, pattern in truthful_26[:10]:
    print(pattern, "Support =", support)

print("\nTop 10 Hallucinated:")
for support, pattern in hallucinated_26[:10]:
    print(pattern, "Support =", support)

===== LAYER 26 =====

Truthful patterns found: 1008
Hallucinated patterns found: 1046

Top 10 Truthful:
[0, 0] Support = 25
[0, 0, 0] Support = 25
[0, 0, 0, 0] Support = 25
[0, 0, 9] Support = 24
[0, 0, 0, 0, 9] Support = 24
[0, 0, 0, 9] Support = 24
[0, 9] Support = 24
[0, 0, 9, 10] Support = 23
[0, 0, 0, 0, 10] Support = 23
[0, 0, 0, 9, 10] Support = 23

Top 10 Hallucinated:
[0, 0] Support = 22
[0, 0, 0] Support = 22
[0, 9] Support = 22
[0, 0, 0, 0] Support = 21
[0, 0, 0, 0, 0] Support = 20
[0, 0, 0, 9] Support = 20
[0, 0, 9] Support = 20
[9, 10] Support = 20
[3, 0] Support = 19
[0, 0, 0, 0, 9] Support = 19


In [6]:
# STEP 16: Calculate hallucination-associated pattern metrics

def contains_subsequence(sequence, pattern):
    """Check whether pattern occurs as an ordered subsequence."""
    j = 0

    for state in sequence:
        if j < len(pattern) and state == pattern[j]:
            j += 1

        if j == len(pattern):
            return True

    return False


# Combine all candidate Layer-26 patterns
candidate_patterns = set()

for support, pattern in truthful_26:
    candidate_patterns.add(tuple(pattern))

for support, pattern in hallucinated_26:
    candidate_patterns.add(tuple(pattern))


results = []

N_TRUTH = len(truthful_sequences)       # 25
N_HALL = len(hallucinated_sequences)    # 25
N_TOTAL = N_TRUTH + N_HALL              # 50


for pattern_tuple in candidate_patterns:

    pattern = list(pattern_tuple)

    truth_support = sum(
        contains_subsequence(seq, pattern)
        for seq in truthful_sequences
    )

    hall_support = sum(
        contains_subsequence(seq, pattern)
        for seq in hallucinated_sequences
    )

    total_support = truth_support + hall_support

    # P(Hallucinated | Pattern)
    confidence = (
        hall_support / total_support
        if total_support > 0 else 0
    )

    # Baseline hallucination probability = 25/50 = 0.5
    baseline_hall = N_HALL / N_TOTAL

    lift = (
        confidence / baseline_hall
        if baseline_hall > 0 else 0
    )

    hall_rate = hall_support / N_HALL
    truth_rate = truth_support / N_TRUTH

    # Positive = more common in hallucinated responses
    contrast = hall_rate - truth_rate

    results.append({
        "Pattern": str(pattern),
        "Pattern_Length": len(pattern),
        "Truthful_Support": truth_support,
        "Hallucinated_Support": hall_support,
        "Truthful_Rate": truth_rate,
        "Hallucinated_Rate": hall_rate,
        "Confidence_Hallucination": confidence,
        "Lift": lift,
        "Contrast": contrast
    })


pattern_metrics = pd.DataFrame(results)

# Keep patterns with reasonable hallucinated support
hall_patterns = pattern_metrics[
    pattern_metrics["Hallucinated_Support"] >= 5
].copy()

# Rank primarily by contrast, then lift/support
hall_patterns = hall_patterns.sort_values(
    by=[
        "Contrast",
        "Lift",
        "Hallucinated_Support"
    ],
    ascending=[False, False, False]
)

print("Candidate patterns:", len(pattern_metrics))
print("Patterns after support filter:", len(hall_patterns))

print("\nTop hallucination-associated patterns:")

display(
    hall_patterns[
        [
            "Pattern",
            "Truthful_Support",
            "Hallucinated_Support",
            "Confidence_Hallucination",
            "Lift",
            "Contrast"
        ]
    ].head(20)
)

Candidate patterns: 1391
Patterns after support filter: 1046

Top hallucination-associated patterns:


,Pattern,Truthful_Support,Hallucinated_Support,Confidence_Hallucination,Lift,Contrast
858,"[3, 4, 4, 4, 4]",3,13,0.812500,1.625000,0.40
415,"[4, 3, 3, 0]",1,10,0.909091,1.818182,0.36
1255,"[4, 3, 3]",1,10,0.909091,1.818182,0.36
153,"[3, 4, 4, 4, 9]",4,13,0.764706,1.529412,0.36
212,"[3, 4, 4, 4]",4,13,0.764706,1.529412,0.36
500,"[3, 4, 4, 4, 0]",4,13,0.764706,1.529412,0.36
1247,"[3, 3]",6,15,0.714286,1.428571,0.36
1388,"[3, 3, 0]",6,15,0.714286,1.428571,0.36
511,"[4, 3, 3, 9]",1,9,0.900000,1.800000,0.32
901,"[4, 3, 3, 0, 9]",1,9,0.900000,1.800000,0.32


In [7]:
# STEP 17: Build final hallucination-associated pattern library

final_patterns = pattern_metrics[
    (pattern_metrics["Hallucinated_Support"] >= 5) &
    (pattern_metrics["Confidence_Hallucination"] >= 0.70) &
    (pattern_metrics["Lift"] >= 1.20) &
    (pattern_metrics["Contrast"] >= 0.20)
].copy()

final_patterns = final_patterns.sort_values(
    by=[
        "Contrast",
        "Lift",
        "Hallucinated_Support"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

final_patterns.insert(
    0,
    "Pattern_ID",
    ["HP_" + str(i + 1).zfill(3)
     for i in range(len(final_patterns))]
)

final_patterns.insert(1, "Layer_ID", 26)

print("Final hallucination-associated patterns:",
      len(final_patterns))

print("\nTop 20 final patterns:")

display(
    final_patterns[
        [
            "Pattern_ID",
            "Layer_ID",
            "Pattern",
            "Truthful_Support",
            "Hallucinated_Support",
            "Confidence_Hallucination",
            "Lift",
            "Contrast"
        ]
    ].head(20)
)

Final hallucination-associated patterns: 182

Top 20 final patterns:


,Pattern_ID,Layer_ID,Pattern,Truthful_Support,Hallucinated_Support,Confidence_Hallucination,Lift,Contrast
0,HP_001,26,"[3, 4, 4, 4, 4]",3,13,0.812500,1.625000,0.40
1,HP_002,26,"[4, 3, 3, 0]",1,10,0.909091,1.818182,0.36
2,HP_003,26,"[4, 3, 3]",1,10,0.909091,1.818182,0.36
3,HP_004,26,"[3, 4, 4, 4, 9]",4,13,0.764706,1.529412,0.36
4,HP_005,26,"[3, 4, 4, 4]",4,13,0.764706,1.529412,0.36
5,HP_006,26,"[3, 4, 4, 4, 0]",4,13,0.764706,1.529412,0.36
6,HP_007,26,"[3, 3]",6,15,0.714286,1.428571,0.36
7,HP_008,26,"[3, 3, 0]",6,15,0.714286,1.428571,0.36
8,HP_009,26,"[4, 3, 3, 9]",1,9,0.900000,1.800000,0.32
9,HP_010,26,"[4, 3, 3, 0, 9]",1,9,0.900000,1.800000,0.32


In [8]:
# STEP 18: Save pattern library + build early-warning matcher

import os
import ast
import json

OUTPUT_DIR = "/content/pattern_mining_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save final pattern library
final_patterns.to_csv(
    f"{OUTPUT_DIR}/hallucination_pattern_library.csv",
    index=False
)

# Convert pattern string -> Python list
pattern_library = []

for _, row in final_patterns.iterrows():
    pattern_library.append({
        "Pattern_ID": row["Pattern_ID"],
        "Layer_ID": int(row["Layer_ID"]),
        "Pattern": ast.literal_eval(row["Pattern"]),
        "Confidence": float(row["Confidence_Hallucination"]),
        "Lift": float(row["Lift"]),
        "Contrast": float(row["Contrast"])
    })


def early_warning_match(sequence, library):
    """
    Check whether a partial activation-state sequence
    contains any hallucination-associated pattern.
    """

    matches = []

    for item in library:

        if contains_subsequence(
            sequence,
            item["Pattern"]
        ):
            matches.append(item)

    # Highest-confidence match first
    matches.sort(
        key=lambda x: (
            x["Confidence"],
            x["Lift"],
            x["Contrast"]
        ),
        reverse=True
    )

    return matches


print("Pattern library saved.")
print("Patterns available:", len(pattern_library))

# Quick sanity test
test_sequence = [3, 4, 4, 4, 4]

matches = early_warning_match(
    test_sequence,
    pattern_library
)

print("\nTest sequence:", test_sequence)
print("Patterns matched:", len(matches))

if matches:
    print("Top match:", matches[0])

Pattern library saved.
Patterns available: 182

Test sequence: [3, 4, 4, 4, 4]
Patterns matched: 3
Top match: {'Pattern_ID': 'HP_001', 'Layer_ID': 26, 'Pattern': [3, 4, 4, 4, 4], 'Confidence': 0.8125, 'Lift': 1.625, 'Contrast': 0.4}


In [9]:
# STEP 19: Test early-warning detection on actual responses

layer26_df = df[df["Layer_ID"] == 26].copy()

early_results = []

for _, row in layer26_df.iterrows():

    prompt_id = int(row["Prompt_ID"])
    true_label = int(row["Response_Label"])
    sequence = row["State_Sequence"]

    first_warning_step = None
    top_match = None

    # Grow partial sequence token-by-token
    for step in range(2, len(sequence) + 1):

        partial_sequence = sequence[:step]

        matches = early_warning_match(
            partial_sequence,
            pattern_library
        )

        if matches:
            first_warning_step = step
            top_match = matches[0]
            break

    warning = first_warning_step is not None

    early_results.append({
        "Prompt_ID": prompt_id,
        "True_Label": true_label,
        "Sequence_Length": len(sequence),
        "Warning": warning,
        "First_Warning_Step": first_warning_step,
        "Warning_Fraction": (
            first_warning_step / len(sequence)
            if warning else None
        ),
        "Matched_Pattern_ID": (
            top_match["Pattern_ID"]
            if top_match else None
        ),
        "Matched_Confidence": (
            top_match["Confidence"]
            if top_match else None
        )
    })


early_results_df = pd.DataFrame(early_results)

print("Responses tested:", len(early_results_df))

print("\nWarnings generated:")
print(early_results_df["Warning"].value_counts())

print("\nWarnings by true label:")
print(
    pd.crosstab(
        early_results_df["True_Label"],
        early_results_df["Warning"]
    )
)

print("\nAverage first warning step:")
print(
    early_results_df[
        early_results_df["Warning"]
    ]["First_Warning_Step"].mean()
)

print("\nAverage warning fraction:")
print(
    early_results_df[
        early_results_df["Warning"]
    ]["Warning_Fraction"].mean()
)

print("\nSample results:")
display(early_results_df.head(10))

Responses tested: 50

Warnings generated:
Warning
True     30
False    20
Name: count, dtype: int64

Warnings by true label:
Warning     False  True 
True_Label              
0              14     11
1               6     19

Average first warning step:
21.466666666666665

Average warning fraction:
0.39088432282913105

Sample results:


,Prompt_ID,True_Label,Sequence_Length,Warning,First_Warning_Step,Warning_Fraction,Matched_Pattern_ID,Matched_Confidence
0,19,0,64,True,43.0,0.671875,HP_020,0.722222
1,25,0,41,True,3.0,0.073171,HP_020,0.722222
2,52,0,75,True,59.0,0.786667,HP_011,0.785714
3,62,1,27,True,4.0,0.148148,HP_031,0.750000
4,75,1,80,True,3.0,0.037500,HP_020,0.722222
5,83,0,49,False,NaN,NaN,None,NaN
6,84,1,79,True,8.0,0.101266,HP_053,0.769231
7,105,0,50,False,NaN,NaN,None,NaN
8,111,0,54,False,NaN,NaN,None,NaN
9,140,1,50,True,32.0,0.640000,HP_011,0.785714


In [10]:
# STEP 20: Early-warning evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_true = early_results_df["True_Label"].astype(int)

# Warning=True means predicted hallucination
y_pred = early_results_df["Warning"].astype(int)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

cm = confusion_matrix(y_true, y_pred)

print("===== EARLY-WARNING PERFORMANCE =====")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nTN =", cm[0, 0])
print("FP =", cm[0, 1])
print("FN =", cm[1, 0])
print("TP =", cm[1, 1])

===== EARLY-WARNING PERFORMANCE =====
Accuracy : 0.6600
Precision: 0.6333
Recall   : 0.7600
F1 Score : 0.6909

Confusion Matrix:
[[14 11]
 [ 6 19]]

TN = 14
FP = 11
FN = 6
TP = 19


In [11]:
# STEP 21: Save final sequential-pattern-mining results

import os
import json

OUTPUT_DIR = "/content/pattern_mining_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. All pattern metrics
pattern_metrics.to_csv(
    f"{OUTPUT_DIR}/pattern_metrics.csv",
    index=False
)

# 2. Final filtered pattern library
final_patterns.to_csv(
    f"{OUTPUT_DIR}/hallucination_pattern_library.csv",
    index=False
)

# 3. Early-warning results for 50 responses
early_results_df.to_csv(
    f"{OUTPUT_DIR}/early_warning_results.csv",
    index=False
)

# 4. Overall evaluation summary
summary = {
    "layer_used": 26,
    "total_responses": 50,
    "truthful_responses": 25,
    "hallucinated_responses": 25,
    "final_pattern_count": int(len(final_patterns)),
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "true_negative": int(cm[0, 0]),
    "false_positive": int(cm[0, 1]),
    "false_negative": int(cm[1, 0]),
    "true_positive": int(cm[1, 1]),
    "average_warning_fraction": float(
        early_results_df.loc[
            early_results_df["Warning"],
            "Warning_Fraction"
        ].mean()
    ),
    "evaluation_type": "proof-of-concept / in-sample",
    "limitation": (
        "Patterns were discovered and evaluated on the same 50 responses; "
        "therefore these metrics are not an unbiased estimate of "
        "generalization performance."
    )
}

with open(
    f"{OUTPUT_DIR}/evaluation_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=4)

print("✅ Final results saved!")

print("\nFiles:")
for file in sorted(os.listdir(OUTPUT_DIR)):
    print("-", file)

✅ Final results saved!

Files:
- early_warning_results.csv
- evaluation_summary.json
- hallucination_pattern_library.csv
- pattern_metrics.csv


In [12]:
import shutil

zip_path = shutil.make_archive(
    "/content/pattern_mining_outputs",
    "zip",
    "/content/pattern_mining_outputs"
)

print("✅ ZIP created:", zip_path)

✅ ZIP created: /content/pattern_mining_outputs.zip
